# N14 · Context Parallel & Ring Attention

**关联 lab**: L04.5

**学习目标**: 用 numpy 手算 ring attention 的累加路径，验证 "Q stationary, K/V rotate" 与单卡 attention 数值等价。理解 CP=2 / CP=4 时通信代价、partial softmax 重归一的不可省略性。

**No-GPU 可完成度**: 100%。

**对应 MiniInfra**: `mini_infra/megatron/core/context_parallel/ring_attention.py`

**对应真实源码**: `github_repo/Megatron-LM/megatron/core/transformer/attention.py`

## 1. CP 是第四并行轴

在 TP/PP/DP 之外，CP（Context Parallel）按 **seq 维度** 切：seq=16K, CP=4 → 每个 rank 处理 4K 长的 chunk。

为什么需要 CP？
- TP 切 hidden 与 head：seq 维不变
- PP 切 layers：seq 维不变
- DP 复制模型：每个 rank 仍处理完整 seq
- **CP 切 seq**：唯一能减少单 rank seq 内存的轴

Attention 是唯一需要跨 seq 通信的算子（每个 Q 要看所有 K）。这就是 ring attention 的存在理由。

## 2. Ring 模型：Q stationary, K/V rotate

CP=N 时，每个 rank 持有：
- 自己 chunk 的 Q（固定不动）
- 自己 chunk 的 K/V（每步 ring 传给下一个 rank）

走 N 步：第 i 步 rank r 算 (q_r, kv_{r-i mod N}) 的 partial attention。

用 `ring_attention_plan` 看具体路径：

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_infra.megatron.core.context_parallel.ring_attention import (
    ring_attention_plan, attention_memory_gb, seqlen_sweep
)

plan = ring_attention_plan(seq_len=16, cp_size=4, hidden_size=4, dtype_bytes=2)
print(f'cp_size={plan["cp_size"]}, q_stationary={plan["q_stationary"]}')
print(f'共 {len(plan["steps"])} 个 (step, q_rank, kv_rank) tuple')
print()
for step in plan['steps']:
    print(f"  step {step['step']}  q_rank={step['q_rank']}  "
          f"kv_rank={step['kv_rank']}  seq=[{step['seq_start']}:{step['seq_end']}]")

**关键观察**: 
- 共 cp_size² 个 (q_rank, kv_rank) 对（每个 q_rank 与每个 kv_rank 都要算一次 partial）
- 共 cp_size 步 ring rotation（每步 N 个 rank 同时进行）
- step 0 时每个 rank 算自己的 (q_r, kv_r)（本地，无通信）
- 之后每步 rotate 一次

## 3. 显存账：CP 切了多少

Naive attention 的 logits 矩阵是 seq × seq，CP 后每个 rank 只持有 chunk × seq 的部分（只有自己的 Q 行）。

In [ ]:
for seq in [4096, 16384, 65536]:
    naive_gb = attention_memory_gb(seq, hidden_size=4096, dtype_bytes=2)
    for cp in [1, 2, 4]:
        local_seq = seq // cp
        local_naive = attention_memory_gb(local_seq, hidden_size=4096, dtype_bytes=2)
        print(f'seq={seq:>5}  CP={cp}  '
              f'naive={naive_gb:>6.2f} GB  '
              f'per-rank-naive={local_naive:>6.2f} GB  '
              f'per-rank-flash={local_seq * 4096 * 2 / 1e9:.4f} GB')
    print()

**结论**: 
- 64K naive attention 单卡需 17 GB（仅 logits），4090 装不下
- CP=4 后每 rank naive 仍需 1.07 GB（4K chunk）
- 真正的救星是 flash attention（O(seq) 而不是 O(seq²)）：64K + flash + CP=4 时每 rank 只需 0.13 GB

## 4. 通信代价 sweep

用 `seqlen_sweep` 把 seq=4K/16K/64K × CP=1/2/4 全跑一遍：

In [ ]:
rows = seqlen_sweep()
print(f'{"seq":>6} {"cp":>4} {"per-rank-mem":>14} {"comm_bytes":>14} {"step_ms":>10}')
for row in rows:
    print(f'{row["seq_len"]:>6} {row["cp_size"]:>4} '
          f'{row["peak_mem_gb_cp"]:>12.2f} GB '
          f'{row["attn_comm_bytes"]/1e6:>12.1f} MB '
          f'{row["step_time_ms"]:>10.2f}')

**观察**: 
- seq 固定时，CP↑ 让每 rank 显存↓，但通信字节↑（与 N-1 成正比）
- 4K seq 上 CP=4 没意义（通信代价 > 计算代价）
- 64K seq 上 CP=4 几乎必须（否则 OOM）

**找 CP 甜区的方法**: 在你的硬件上跑 sweep，比较 step_time_ms。一般 16K-32K 是 CP=2 的甜区。

## 5. Partial softmax 累加（数值等价证明）

Ring attention 的难点：每 rank 只看到部分 K，必须 partial softmax 累加，最后用 final_max 重归一。

用最小例子（4 token, CP=2）手算一遍：

In [ ]:
import math

# 4 token, hidden=2，假设 attention scores 已经是 logits
logits = [
    [1.0, 0.5, 2.0, -1.0],   # token 0 vs all tokens
    [0.0, 1.5, 1.0, 0.5],    # token 1 vs all tokens
    [2.0, 0.0, 1.0, 0.5],    # token 2 vs all tokens
    [0.5, 1.0, 0.5, 2.0],    # token 3 vs all tokens
]
v = [1.0, 2.0, 3.0, 4.0]  # values

def reference_attention(logits_row, v):
    m = max(logits_row)
    e = [math.exp(l - m) for l in logits_row]
    z = sum(e)
    return sum(ei * vi for ei, vi in zip(e, v)) / z

ref_out = [reference_attention(row, v) for row in logits]
print('Reference (single-card) attention output:', [round(o, 4) for o in ref_out])

In [ ]:
# CP=2: rank 0 持有 token 0,1; rank 1 持有 token 2,3
# Ring step 0: 每 rank 算自己的 partial (本地 Q × 本地 K/V)
# Ring step 1: K/V rotate, 每 rank 算 (本地 Q × 对方 K/V)

def partial(logits_row, v_chunk, indices):
    sub_logits = [logits_row[i] for i in indices]
    sub_v = [v_chunk[i] for i in range(len(indices))]
    m = max(sub_logits)
    e = [math.exp(l - m) for l in sub_logits]
    s = sum(e)
    o = sum(ei * vi for ei, vi in zip(e, sub_v))
    return m, s, o

# rank 0 视角，token 0 (q_local=0)
# step 0: 看 token 0,1 (本地 K)
m0_local, s0_local, o0_local = partial(logits[0], [v[0], v[1]], indices=[0, 1])
# step 1: 看 token 2,3 (rotate 来的)
m0_remote, s0_remote, o0_remote = partial(logits[0], [v[2], v[3]], indices=[2, 3])

print(f'token 0 partial step0: max={m0_local}, sum={s0_local:.4f}, partial_out={o0_local:.4f}')
print(f'token 0 partial step1: max={m0_remote}, sum={s0_remote:.4f}, partial_out={o0_remote:.4f}')

# 重归一：用 final_max
final_max = max(m0_local, m0_remote)
rescale_local = math.exp(m0_local - final_max)
rescale_remote = math.exp(m0_remote - final_max)
final_sum = s0_local * rescale_local + s0_remote * rescale_remote
final_out = (o0_local * rescale_local + o0_remote * rescale_remote) / final_sum

print(f'\ntoken 0 ring final: {final_out:.4f}')
print(f'token 0 reference : {ref_out[0]:.4f}')
print(f'max_abs_err = {abs(final_out - ref_out[0]):.2e}  (应 < 1e-12)')

**数值等价！** Ring attention 与单卡 attention 在数值上完全一致。

**关键不变量**: 必须用 final_max 重归一所有 partial。如果直接 `sum(o_i / s_i)` 加起来 → 错误。

## 6. 反证：漏 final_max 重归一会怎样

In [ ]:
# 错误版：直接把 partial output 平均
buggy_out = (o0_local / s0_local + o0_remote / s0_remote) / 2
print(f'buggy        = {buggy_out:.4f}')
print(f'reference    = {ref_out[0]:.4f}')
print(f'buggy err    = {abs(buggy_out - ref_out[0]):.4f}  (远 > final 版本)')

在 fp32 + 短 seq 下偏差不大，但 fp16 + 长 seq + 累积多步 ring 后会让 loss NaN。

对应 ticket: `cp_shape_mismatch_001`。

## 7. 自检问题

1. CP=4 一个 step 总共需要几次 K/V 传递？通信总字节数与 seq 是什么关系？
2. 为什么是 "Q stationary, K/V rotate" 而不是反过来？（提示：哪个张量在 attention 中需要看所有位置）
3. 4K seq 上 CP=4 是个好主意吗？为什么？
4. Ring attention 的 partial softmax 累加为什么必须用 final_max 重归一？省略会怎样？
5. CP=2 + flash-attn vs CP=1 + 普通 attention，64K seq 上哪个显存更省？

## 8. 与 lab 对接

Lab 任务：在 Megatron 上跑 seq=4K/16K/64K × CP=1/2/4 矩阵。

前置 prediction（写入 `prediction.yaml`）：
- CP=2 在 16K 上 step_time 优于 CP=1 + recompute=full
- attn_comm_bytes 与 seq 成线性，与 CP 成反比（用 ring_attention_plan 验证）
- ring 累加与单卡 max_abs_err < 1e-5

Smoke：
```bash
torchrun --nproc_per_node=2 labs/l09_long_context_cp/scripts/run_seqlen.py \
  --seq 16384 --cp 2 --recompute selective
```

失败时优先查 `cp_shape_mismatch_001.yaml`（mask 切分 + ring 步数 + partial 累加）。